## DX702 Coding Homework ##
Keshia-Lee Martin<br>
October 11, 2026

### Week 1

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

hw1_1 = pd.read_csv("homework_1.1.csv")
hw1_2 = pd.read_csv("homework_1.2.csv")

Question 1: Which of the following is closest to the coefficient of X1?

In [ ]:
model = smf.ols("Y ~ X1 + X2 + X3", data=hw1_1).fit()
coeff_x1 = model.params["X1"]
print(round(coeff_x1, 3))

1.007


Question 2: Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi's), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset on average (not fixing the other Xi's)?
Hint: For the former, regress Y on Xi alone. For the latter, regress Y on all three Xi's.


In [ ]:
only_x1   = smf.ols("Y ~ X1", data=hw1_1).fit()
only_x2   = smf.ols("Y ~ X2", data=hw1_1).fit()
only_x3   = smf.ols("Y ~ X3", data=hw1_1).fit()

coeff_x1 = model.params["X1"]
coeff_x2 = model.params["X2"]
coeff_x3 = model.params["X3"]

coeff_x1_alone = only_x1.params["X1"]
coeff_x2_alone = only_x2.params["X2"]
coeff_x3_alone = only_x3.params["X3"]

print("X1:", round(coeff_x1, 3), " alone", round(coeff_x1_alone, 3),
      " diff", round(coeff_x1_alone - coeff_x1, 3))
print("X2:", round(coeff_x2, 3), " alone", round(coeff_x2_alone, 3),
      " diff", round(coeff_x2_alone - coeff_x2, 3))
print("X3:", round(coeff_x3, 3), " alone", round(coeff_x3_alone, 3),
      " diff", round(coeff_x3_alone - coeff_x3, 3))

X1: 1.007  alone 1.842  diff 0.835
X2: 1.965  alone 4.084  diff 2.119
X3: 2.975  alone 3.097  diff 0.122


Question 3: When regressing Y on all Xi's together, which coefficient is most significant, taking the t-statistic as the measure of significance?

In [12]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 11 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:43:50   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0026      0.016      0.166      0.8

Question 4: What is the distance of the farthest match in this set?

In [17]:
treated = hw1_2[hw1_2["X"] == 1].reset_index(drop=True)
control = hw1_2[hw1_2["X"] == 0].reset_index(drop=True)

n_neighbor = NearestNeighbors(n_neighbors=1).fit(control[["Z"]])
distances, positions = n_neighbor.kneighbors(treated[["Z"]])
 
print("Treated units =", len(treated))
print("Farthest match distance =", distances.max())


Treated units = 48
Farthest match distance = 0.2102170871093757


Question 5: What is the effect? The difference between the average Y for the X = 0 values and the average Y for the X = 1 values, where the X = 0 sample is the set of best matches — so the matched sample of X = 0 against the full sample of X = 1.

In [18]:
matched = control.iloc[positions.flatten()]

print("mean Y treated =", treated["Y"].mean())
print("mean Y matched control =", matched["Y"].mean())
print("Effect =", treated["Y"].mean() - matched["Y"].mean())


mean Y treated = 1.125597137873841
mean Y matched control = 0.582237072655257
Effect = 0.5433600652185839


Question 6: How many duplicates do you end up with? Count all but the first in each group. One way to do this is radius_neighbors.

In [20]:
pairs_total = 0
controls_used = set()

n_neighbor2 = NearestNeighbors(radius=0.2).fit(control[["Z"]])
distances, groups = n_neighbor2.radius_neighbors(treated[["Z"]])

for group in groups:
    pairs_total = pairs_total + len(group)
    for position in group:
        controls_used.add(int(position))

print(pairs_total)
print(len(controls_used))
print(pairs_total - len(controls_used))


737
52
685


Question 7: What is the effect?
